# 01 · Data

Download the universe, check data quality and get a first look at prices, correlations and funding.

We trade **perpetual futures** (Binance USDⓈ-M), because shorting one leg of a pair is only practical with perps.
That gives us two data sets per coin: hourly prices and the funding rate history (paid every 8h).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Make the repo root importable, whether Jupyter starts in notebooks/ or the repo root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import data, strategy, backtest, plots

plots.apply_style()
pd.set_option("display.float_format", "{:.4f}".format)
cfg = data.load_config()

## 1. Load config

`config.yaml` stores the universe as base coins; `data.perp_symbol` maps them to the ccxt perpetual symbol.
Dates are whole days and inclusive: train = [start, train_end], test = (train_end, end].

In [ ]:
setup = cfg["setup"]
print(f"Exchange:  {setup['exchange']}")
print(f"Universe:  {[data.perp_symbol(c) for c in setup['universe']]}")
print(f"Timeframe: {setup['timeframe']}")
print(f"Period:    {setup['start']} to {setup['end']}  (train until {setup['train_end']})")
print(f"Costs:     fee {setup['fee_per_leg']:.2%} + slippage {setup['slippage_per_leg']:.2%} per leg and trade")

## 2. Download / load data

`data.download_universe` skips files that are cached **and** cover the configured date range; otherwise it
downloads them (the first download takes about 6 minutes). `data.load_prices` puts every coin on one complete
hourly UTC grid, forward-fills gaps of up to `max_ffill_bars` bars and drops coins with more than
`max_missing_pct` missing bars. `data.load_funding` puts the funding rates on the same grid (rate at each
funding hour, 0 otherwise).

In [ ]:
data.download_universe(cfg)  # no-op if all files are cached and cover the date range

prices = data.load_prices(cfg, verbose=False)
funding = data.load_funding(cfg, coins=list(prices.columns))
train_prices, test_prices = data.split_train_test(prices, setup["train_end"])

print(f"Prices: {prices.shape[0]} hourly bars x {prices.shape[1]} coins")
print(f"Train:  {train_prices.index[0]} to {train_prices.index[-1]} ({len(train_prices)} bars)")
print(f"Test:   {test_prices.index[0]} to {test_prices.index[-1]} ({len(test_prices)} bars)")
prices.tail(3)

## 3. Data quality

Raw data before cleaning: available bars, missing bars on the full hourly grid, the longest gap,
and the first/last timestamp per coin. We also look at the largest 1h move per coin as an outlier check
(a huge spike that immediately reverts would point to a bad tick).

In [ ]:
quality = data.data_quality(data.load_raw_closes(cfg))

log_ret = np.log(prices).diff()
quality["max_abs_1h_return"] = log_ret.abs().max()
quality["time_of_max_move"] = log_ret.abs().idxmax()

quality.style.format({"missing_pct": "{:.2%}", "max_abs_1h_return": "{:.1%}"})

## 4. Price overview

All coins rebased to 100 at the start, on a log scale so that equal percentage moves look equal.
Six coins are highlighted, the rest are grey. The dashed line marks the end of the train period.

In [ ]:
plots.plot_prices(prices, highlight=["BTC", "ETH", "SOL", "BNB", "XRP", "LTC"], train_end=setup["train_end"],
                  save="01_prices.png")

## 5. Correlations

Correlation of hourly **log returns** on the **train period only** (the test period stays untouched).
We use returns, not price levels: two trending price series look highly correlated even when unrelated
(spurious correlation).

In [ ]:
train_ret = np.log(train_prices).diff().dropna()
corr = train_ret.corr()
plots.plot_correlation_matrix(corr, title="Correlation of hourly log returns (train period)",
                              save="01_correlation.png")

In [ ]:
# Top pairs by correlation (every pair once)
strategy.correlation_screen(train_prices, top_n=10)

## 6. Funding rates

Holders of a perpetual pay or receive funding every 8h: a **positive** rate means longs pay shorts.
In a pair trade we are long one leg and short the other, so funding is a cost on one leg and income
on the other. Annualized funding = average funding per hour × 24 × 365.

In [ ]:
train_funding, test_funding = data.split_train_test(funding, setup["train_end"])
hours_per_year = 24 * 365
annual_funding = pd.DataFrame({
    "full period": funding.mean() * hours_per_year,
    "train": train_funding.mean() * hours_per_year,
    "test": test_funding.mean() * hours_per_year,
}).sort_values("full period", ascending=False)
annual_funding.index.name = "Coin"

display(plots.plot_bars(annual_funding["full period"] * 100,
                        title="Average annualized funding rate (paid by longs), full period",
                        ylabel="% per year"))
annual_funding.style.format("{:.2%}")

In [ ]:
# Funding over time for three coins: rolling 30-day mean, annualized, to see regimes rather than noise
coins_to_plot = ["BTC", "ETH", "BNB"]
rolling_annual = funding[coins_to_plot].rolling("30D").mean() * hours_per_year * 100
plots.plot_series(rolling_annual, title="Funding rate, 30-day rolling mean, annualized",
                  ylabel="% per year", hline=0, vline=setup["train_end"])

## 7. Observations

- **Data quality is clean.** All 12 perpetuals have the full 33,600 hourly bars from 2023-01-01 to 2026-10-31 with no missing bars, so no forward-filling was needed and no coin is dropped. Funding comes every 8h for every coin (4,200 events each).
- **Large moves are real, not bad ticks.** The biggest 1h moves cluster on known market-wide events (e.g. the crash of 10 Oct 2025 for LINK, DOT and LTC, 3 Feb 2025 for ETH and BNB), so we keep them. They are exactly the moments when a pair's spread can blow out, which is what the z-stop is for.
- **Prices diverge strongly in levels.** SOL rose ~12x over the train period while DOT lost more than half its value. Level correlation alone says little, so pair selection must rely on return correlation plus a cointegration test.
- **Return correlations are high across the board** (median 0.68 across all 66 pairs on train data, half of them between 0.63 and 0.72), driven by a common crypto market factor. The most correlated pairs are BTC–ETH (0.81) and a large-cap altcoin cluster (ADA, DOT, LINK, AVAX, ETH). These are natural candidates for notebook 02. TRX is the least correlated coin (about 0.45 with everything else).
- **Funding is material and regime-dependent.** Longs paid about 5–10% per year on most coins in the train period, with peaks near 40% per year in early 2024. BNB and TRX are the exceptions with near-zero or negative funding. In the test period funding is much lower (mostly 0–5%). For a market-neutral pair, the two legs' funding partly cancels, but a pair with very different funding (e.g. long BNB / short ETH) earns or pays the funding spread every 8h, so the backtest must include it.